# DSA Week 8 -- Performance Sprint 2

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Focus:** Benchmark baseline vs optimized, create plots and report

## Learning Objectives

1. Run a full benchmark suite at 5+ input sizes
2. Create comparison plots (time + speedup)
3. Compute and interpret speedup factors
4. Generate benchmark_results.json and benchmark_plot.png
5. Confirm >= 1.5x speedup target

## Deliverables

By the end of this session you must have:
- `reports/benchmark/benchmark_results.json`
- `reports/benchmark/benchmark_plot.png`
- Speedup >= 1.5x at the largest input size

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: The Complete Benchmark Template

This is the template you will adapt for your project.

In [ ]:
import timeit
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def run_full_benchmark(baseline_func, optimized_func, data_gen,
                       sizes, n_runs=50, label="Benchmark"):
    """Run a complete benchmark comparison.

    Returns results dict suitable for JSON export.
    """
    results = {
        "description": label,
        "sizes": [],
        "baseline_ms": [],
        "optimized_ms": [],
        "speedup": [],
    }

    print("=== " + label + " ===")
    print()
    print("  " + "n".rjust(10) + " | " + "baseline".rjust(12) + " | " + "optimized".rjust(12) + " | " + "speedup".rjust(8))
    print("  " + "-" * 10 + "-|-" + "-" * 12 + "-|-" + "-" * 12 + "-|-" + "-" * 8)

    for n in sizes:
        data = data_gen(n)

        t_base = timeit.timeit(lambda d=data: baseline_func(d), number=n_runs)
        t_opt = timeit.timeit(lambda d=data: optimized_func(d), number=n_runs)

        base_ms = t_base / n_runs * 1000
        opt_ms = t_opt / n_runs * 1000
        speedup = base_ms / opt_ms if opt_ms > 0 else float("inf")

        results["sizes"].append(n)
        results["baseline_ms"].append(round(base_ms, 4))
        results["optimized_ms"].append(round(opt_ms, 4))
        results["speedup"].append(round(speedup, 1))

        print("  " + "{:>10,}".format(n) + " | " + "{:>10.3f}ms".format(base_ms) + " | " + "{:>10.3f}ms".format(opt_ms) + " | " + "{:>6.1f}x".format(speedup))

    print()
    max_speedup = max(results["speedup"])
    target_met = max_speedup >= 1.5
    print("  Max speedup: " + str(max_speedup) + "x")
    print("  Target (1.5x): " + ("MET" if target_met else "NOT MET -- keep optimizing!"))
    return results

def save_benchmark(results, json_path, plot_path):
    """Save benchmark results as JSON and create comparison plot."""
    # Save JSON
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print("  Saved: " + json_path)

    # Create plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    sizes = results["sizes"]

    ax1.plot(sizes, results["baseline_ms"], "o-", label="Baseline",
             linewidth=2, color="#F44336", markersize=6)
    ax1.plot(sizes, results["optimized_ms"], "s-", label="Optimized",
             linewidth=2, color="#4CAF50", markersize=6)
    ax1.set_title("Execution Time Comparison", fontsize=14)
    ax1.set_xlabel("Input Size (n)")
    ax1.set_ylabel("Time (ms)")
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)

    colors = ["#4CAF50" if s >= 1.5 else "#FF9800" for s in results["speedup"]]
    ax2.bar(range(len(sizes)), results["speedup"], color=colors)
    ax2.set_xticks(range(len(sizes)))
    ax2.set_xticklabels(["{:,}".format(n) for n in sizes], rotation=45)
    ax2.set_title("Speedup Factor", fontsize=14)
    ax2.set_xlabel("Input Size (n)")
    ax2.set_ylabel("Speedup (x)")
    ax2.axhline(y=1.5, color="red", linestyle="--", label="1.5x target", linewidth=2)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    os.makedirs(os.path.dirname(plot_path), exist_ok=True)
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  Saved: " + plot_path)

---
## Part 2: Run the Demo Benchmark

In [ ]:
import random
random.seed(42)

# Baseline: find duplicates with list
def baseline_dedup(data):
    seen = []
    dupes = []
    for x in data:
        if x in seen:
            dupes.append(x)
        seen.append(x)
    return dupes

# Optimized: find duplicates with set
def optimized_dedup(data):
    seen = set()
    dupes = []
    for x in data:
        if x in seen:
            dupes.append(x)
        seen.add(x)
    return dupes

def gen_data(n):
    return [random.randint(0, n // 2) for _ in range(n)]

# Verify correctness first!
test_data = gen_data(1000)
d1 = baseline_dedup(test_data)
d2 = optimized_dedup(test_data)
assert d1 == d2, "Results must match!"
print("Correctness verified!")
print()

# Run benchmark
sizes = [1_000, 2_000, 5_000, 10_000, 20_000]
results = run_full_benchmark(baseline_dedup, optimized_dedup, gen_data,
                             sizes, n_runs=10, label="Dedup: list vs set")

# Save
save_benchmark(results,
               "reports/benchmark/benchmark_results.json",
               "reports/benchmark/benchmark_plot.png")

---
## Part 3: Generate the Complexity Report

In [ ]:
# Add complexity analysis to the results
results["baseline_complexity"] = "O(n^2) -- list membership is O(n) per check"
results["optimized_complexity"] = "O(n) -- set membership is O(1) per check"
results["data_structure_used"] = "set (hash set)"
results["why_faster"] = "Replaced O(n) list scan with O(1) set lookup for each of n items"

# Re-save with complexity info
with open("reports/benchmark/benchmark_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("=== Complexity Report ===")
print()
print("Baseline:  " + results["baseline_complexity"])
print("Optimized: " + results["optimized_complexity"])
print("DS used:   " + results["data_structure_used"])
print("Why:       " + results["why_faster"])
print()
print("Max speedup at n=" + "{:,}".format(results["sizes"][-1]) + ": " + str(results["speedup"][-1]) + "x")

---
## Part 4: Your Turn

Replace the demo above with YOUR project's baseline vs optimized functions.
Run the benchmark, generate the artifacts, and verify >= 1.5x speedup.

**Checklist:**
- [ ] Correctness verified (baseline and optimized give same results)
- [ ] Benchmark runs at 5+ input sizes
- [ ] `reports/benchmark/benchmark_results.json` exists
- [ ] `reports/benchmark/benchmark_plot.png` exists
- [ ] Max speedup >= 1.5x
- [ ] Complexity analysis included

In [ ]:
# TODO: Your project benchmark
# baseline_func = your_baseline
# optimized_func = your_optimized
# data_gen = your_data_generator
# sizes = [1000, 5000, 10000, 50000, 100000]

# results = run_full_benchmark(baseline_func, optimized_func, data_gen,
#                              sizes, n_runs=20, label="Your Project Benchmark")
# save_benchmark(results,
#                "reports/benchmark/benchmark_results.json",
#                "reports/benchmark/benchmark_plot.png")

print("Replace this with your project benchmark!")

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)